# CLK104 configuration test
PS generates a 250 MHz reference clock
Measuring "unknown" clock from CLK104 board (CLK104_PL_CLK_P/N)
via `freq_count_axi` module (frequency result readable via AXI bus)

Control of CLK104_CLK_SPI_MUX_SEL0/1 is also provided here via AXI_GPIO instance (also routed to LEDs 0 & 7)

All other interface with the CLK104 is via the I2C bus directly from the PS (not present in the overlay).
Use xrfclk or the underlying Linux serial driver to write to the LMK/LMX chips.

TODO: Readback registers.  xrfclk doesn't seem to provide a mechanism to do this.

In [1]:
import clk104if
import importlib
importlib.reload(clk104if)

<module 'clk104if' from '/home/xilinx/jupyter_notebooks/RFSoC-MTS/clk104if/clk104if.py'>

In [7]:
def print_attrs(obj, all=False):
    print("Attrs:")
    for attr in dir(obj):
        if all or (not attr.startswith("_")):
            print("  " + attr)

def print_dict(dd, indent=0, depth=-1):
    ind = " "*indent
    for key, val in dd.items():
        print("{}{}:".format(ind, key), end="")
        if hasattr(val, "keys"):
            if depth == 0:
                print(" dict len {}".format(len(val)))
            else:
                print()
                print_dict(val, indent+2, depth-1)
        else:
            print(" {}".format(val))

def print_dicts(obj, depth=-1):
    for dd, name in ((obj.ip_dict, "ip_dict"), (obj.mem_dict, "mem_dict"), (obj.hierarchy_dict, "hierarchy_dict")):
        print(name + ":")
        print_dict(dd, indent=2, depth=depth)

In [3]:
ol = clk104if.CLK104IF("clk104if.bit")
ol.is_loaded()

True

In [4]:
ol.is_loaded()

True

In [13]:
ol.ip_dict

{'freq_count_axi_0': {'type': 'xilinx.com:module_ref:freq_count_axi:1.0',
  'mem_id': 's_axi',
  'memtype': 'REGISTER',
  'gpio': {},
  'interrupts': {},
  'parameters': {'REF_CLK_DIFF': 'BYPASS',
   'UNK_CLK_DIFF': 'TRUE',
   'C_S_AXI_DATA_WIDTH': '32',
   'C_S_AXI_ADDR_WIDTH': '8',
   'Component_Name': 'clk104if_freq_count_axi_0_0',
   'EDK_IPTYPE': 'PERIPHERAL',
   'C_BASEADDR': '0xA0000000',
   'C_HIGHADDR': '0xA0000FFF',
   'DATA_WIDTH': '32',
   'PROTOCOL': 'AXI4LITE',
   'FREQ_HZ': '99999001',
   'ID_WIDTH': '0',
   'ADDR_WIDTH': '8',
   'AWUSER_WIDTH': '0',
   'ARUSER_WIDTH': '0',
   'WUSER_WIDTH': '0',
   'RUSER_WIDTH': '0',
   'BUSER_WIDTH': '0',
   'READ_WRITE_MODE': 'READ_WRITE',
   'HAS_BURST': '0',
   'HAS_LOCK': '0',
   'HAS_PROT': '1',
   'HAS_CACHE': '0',
   'HAS_QOS': '0',
   'HAS_REGION': '0',
   'HAS_WSTRB': '1',
   'HAS_BRESP': '1',
   'HAS_RRESP': '1',
   'SUPPORTS_NARROW_BURST': '0',
   'NUM_READ_OUTSTANDING': '1',
   'NUM_WRITE_OUTSTANDING': '1',
   'MAX_BURST_L

In [5]:
fc = clk104if.FreqCounter(ol.ip_dict["freq_count_axi_0"])

In [6]:
# GPIO Output of width 2 to control CLK104_CLK_SPI_MUX_SEL0/1
# Note that these outputs to CLK104 are also routed to LEDs: (placed far apart so you can see them clearly on the webcam)
#    CLK104_CLK_SPI_MUX_SEL0 => LED0
#    CLK104_CLK_SPI_MUX_SEL1 => LED7
# PYNQ seems to have a bug recognizing "axi_gpio" instances from the block diagram.  Maybe I need to change the name?
base = 0xA0010000 # Straight from the hwh file
from pynq.lib import AxiGPIO
desc = {
    "phys_addr": base,
    "addr_range": 65536,
    "interrupts": {}
}
gpio = AxiGPIO(desc)
gpio.setlength(2, 1)
gpio.setdirection(AxiGPIO.Output, channel=1)

In [15]:
gpio.channel1.write(3, mask=3)

In [16]:
gpio.channel1.write(0, mask=3)

In [9]:
# Read CLK104 input frequency
print("{:3f} MHz".format(fc.read()*1.0e-6))

500.001997 MHz


In [14]:
# Test all 3 CLK104 mux sel combinations
import time
for sel in range(3):
    gpio.channel1.write(sel, mask=3)
    time.sleep(2)
    print("{}: {:3f} MHz".format(sel, fc.read()*1.0e-6))

0: 500.001907 MHz
1: 500.001922 MHz
2: 500.001922 MHz


In [ ]:
# CLK104 board seems to be operating in "auto" clkin selection mode

## Test readback of register list from xrfclk

In [31]:
import clk104_decode
importlib.reload(clk104_decode)

<module 'clk104_decode' from '/home/xilinx/jupyter_notebooks/RFSoC-MTS/clk104if/clk104_decode.py'>

In [14]:
import xrfclk

In [4]:
xrfclk.set_ref_clks(lmk_freq=500.0, lmx_freq=4000.0)

In [32]:
fdict = clk104_decode.extract_from_xrfclk(xrfclk.xrfclk, 500.0, chip=clk104_decode.CHIP_LMK)
print_dict(fdict)

RESET: (1, 0, 7, 7)
SPI_3WIRE_DIS: (1, 0, 4, 4)
CLKout0_1_ODL: (0, 256, 6, 6)
CLKout0_1_IDL: (0, 256, 5, 5)
DCLKout0_DIV: (6, 256, 0, 4)
DCLKout0_DDLY_CNTH: (5, 257, 4, 7)
DCLKout0_DDLY_CNTL: (5, 257, 0, 3)
DCLKout0_ADLY: (0, 259, 3, 7)
DCLKout0_ADLY_MUX: (0, 259, 2, 2)
DCLKout0_MUX: (1, 259, 0, 1)
DCLKout0_HS: (0, 260, 6, 6)
SDCLKout1_MUX: (1, 260, 5, 5)
SDCLKout1_DDLY: (1, 260, 1, 4)
SDCLKout1_HS: (0, 260, 0, 0)
SDCLKout1_ADLY_EN: (0, 261, 4, 4)
SDCLKout1_ADLY: (0, 261, 0, 3)
DCLKout0_DDLY_PD: (0, 262, 7, 7)
DCLKout0_HSg_PD: (1, 262, 6, 6)
DCLKout0_ADLYg_PD: (1, 262, 5, 5)
DCLKout0_ADLY_PD: (1, 262, 4, 4)
CLKout0_1_PD: (0, 262, 3, 3)
SDCLKout1_DIS_MODE: (0, 262, 1, 2)
SDCLKout1_PD: (0, 262, 0, 0)
SDCLKout1_POL: (0, 263, 7, 7)
CLKout1_FMT: (1, 263, 4, 6)
DCLKout0_POL: (0, 263, 3, 3)
CLKout0_FMT: (1, 263, 0, 2)
CLKout2_3_ODL: (0, 264, 6, 6)
CLKout2_3_IDL: (0, 264, 5, 5)
DCLKout2_DIV: (6, 264, 0, 4)
DCLKout2_DDLY_CNTH: (5, 265, 4, 7)
DCLKout2_DDLY_CNTL: (5, 265, 0, 3)
DCLKout2_ADLY: (0,

In [33]:
fdict = clk104_decode.extract_from_xrfclk(xrfclk.xrfclk, 4000.0, chip=clk104_decode.CHIP_LMX)
print_dict(fdict)

rb_VCO_DACISET: (0, 112, 0, 8)
rb_VCO_CAPCTRL: (0, 111, 0, 7)
rb_LD_VTUNE: (0, 110, 9, 10)
rb_VCO_SEL: (0, 110, 5, 7)
RAMP_TRIG_CAL: (0, 106, 4, 4)
RAMP_SCALE_COUNT: (0, 106, 0, 2)
RAMP_DLY_CNT: (0, 105, 6, 15)
RAMP_MANUAL: (1, 105, 5, 5)
RAMP1_NEXT: (0, 105, 4, 4)
RAMP1_NEXT_TRIG: (1, 105, 0, 1)
RAMP1_LEN: (0, 104, 0, 15)
RAMP1_INC[15:0]: (0, 103, 0, 15)
RAMP1_INC[29:16]: (16256, 102, 0, 13)
RAMP1_DLY: (0, 101, 6, 6)
RAMP1_RST: (0, 101, 5, 5)
RAMP0_NEXT: (1, 101, 4, 4)
RAMP0_NEXT_TRIG: (1, 101, 0, 1)
RAMP0_LEN: (0, 100, 0, 15)
RAMP0_INC[15:0]: (0, 99, 0, 15)
RAMP0_INC[29:16]: (128, 98, 2, 15)
RAMP0_DLY: (0, 98, 0, 0)
RAMP0_RST: (0, 97, 15, 15)
RAMP_TRIGB: (1, 97, 7, 10)
RAMP_TRIGA: (1, 97, 3, 6)
RAMP_BURST_TRIG: (0, 97, 0, 1)
RAMP_BURST_EN: (0, 96, 15, 15)
RAMP_BURST_COUNT: (0, 96, 2, 14)
RAMP_LIMIT_LOW[15:0]: (0, 86, 0, 15)
RAMP_LIMIT_LOW[31:16]: (54016, 85, 0, 15)
RAMP_LIMIT_LOW[32]: (1, 84, 0, 0)
RAMP_LIMIT_HIGH[15:0]: (0, 83, 0, 15)
RAMP_LIMIT_HIGH[31:16]: (7680, 82, 0, 15)
RAMP_L

## Test ipq-pynq-utils
https://github.com/kit-ipq/ipq-pynq-utils

In [34]:
from ipq_pynq_utils import ZCU208Board
board = ZCU208Board()

ModuleNotFoundError: No module named 'spidev'

In [35]:
import os
os.getcwd()

'/home/xilinx/jupyter_notebooks/RFSoC-MTS/clk104if'

In [36]:
!which python

/usr/local/share/pynq-venv/bin/python


In [37]:
!pip install spidev

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for spidev: filename=spidev-3.6-cp310-cp310-linux_aarch64.whl size=42285 sha256=3de682880ac88e420e140391c73109d1478d34a5d1166ec09e36403a26c15ad7
  Stored in directory: /root/.cache/pip/wheels/7d/c1/f4/0413111bca1bfb577a4d40337c4e5ca97ca887315900fe538e
Successfully built spidev
